# 第 3 天 — 对话式 AI — 也就是 Chatbot！

In [ ]:
# 导入

# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
# 导入 Gradio：快速搭建可交互的 Web 演示界面（聊天框、按钮等）
import gradio as gr

In [ ]:
# 从名为 .env 的文件加载环境变量
# 打印密钥前缀以便调试

load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# 初始化

# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()
# 选定本次实验使用的模型名称（model id）
MODEL = 'gpt-4.1-mini'

In [ ]:
# 同样，我会保持科学家模式，在实验中修改这个全局变量

system_message = "You are a helpful assistant"

## 现在，编写一个新的回调

我们现在需要编写一个名为：

`chat(message, history)`

的函数，它将作为我们提供给 Gradio 的回调函数。

### 这个函数的职责

接收一条消息、先前的对话，并返回响应。


In [ ]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    return "bananas"

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

## 好！让我们写一个稍好一点的 chat 回调！

In [ ]:

# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

## 好，继续前进！

用系统消息添加上下文，并给出示例回答……这又是「单次示例提示」（one shot prompting）

In [ ]:
# 系统消息（system message）：聊天场景下的角色设定，等价于 system prompt
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:

# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">对话式助手当然是生成式 AI 极为常见的用例，最新的前沿模型在细腻对话方面表现惊人。Gradio 也让用户界面变得容易。我们还掌握了另一项关键技能：如何用提示词提供上下文、信息和示例。
<br/><br/>
想想如何把 AI 助手应用到你的业务中，并自己做一个原型。用系统提示词给出业务上下文，并为 LLM 设定语气。</span>
        </td>
    </tr>
</table>